# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# Carga el dataset WBCD desde UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
# El dataset no tiene encabezado. Definimos nombres básicos de columnas.
# Columna 0: ID, Columna 1: Diagnosis, Columnas 2-31: Características
columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df_wbcd = pd.read_csv(url, header=None, names=columns)

# Mostrar información general del dataset
print("Dimensiones del dataset:", df_wbcd.shape)
display(df_wbcd.head())

Dimensiones del dataset: (569, 32)


,id,diagnosis,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 2. Preprocesamiento

In [9]:
# Para evitar que la reejecución de esta celda genere NaN al intentar mapear
# un DataFrame que ya ha sido transformado, volvemos a leer de la variable original
# o simplemente validamos si la columna 'diagnosis' aún contiene strings ('M' o 'B')

# Volvemos a inicializar df_wbcd desde el origen para asegurar la consistencia si se reejecuta
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df_wbcd = pd.read_csv(url, header=None, names=columns)

# Convierte la columna de diagnosis (M = Maligno, B = Benigno) a binario (1/0)
df_wbcd['diagnosis'] = df_wbcd['diagnosis'].map({'M': 1, 'B': 0})

# Separa en X (features) e y (target)
X = df_wbcd.drop(columns=['id', 'diagnosis'])
y = df_wbcd['diagnosis']

# Haz el train_test_split (80% train, 20% test) con un random_state para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

Tamaño de X_train: (455, 30)
Tamaño de X_test: (114, 30)


## 3. Escalamiento (Solo para k-NN)

In [5]:
# Inicializa el escalador
scaler = StandardScaler()

# Aplica StandardScaler a X_train y X_test.
# Recuerda: .fit_transform() en train, y solo .transform() en test.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos de entrenamiento escalados correctamente.")

Datos de entrenamiento escalados correctamente.


## 4. Modelado y Matrices de Confusión

In [6]:
# 1. Entrena un KNeighborsClassifier(n_neighbors=5) con los datos escalados
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# 2. Entrena un DecisionTreeClassifier(max_depth=4) con los datos SIN escalar
# Los árboles de decisión no son sensibles a la escala de las características
dtree = DecisionTreeClassifier(max_depth=4, random_state=42)
dtree.fit(X_train, y_train)
y_pred_dtree = dtree.predict(X_test)

# 3. Imprime las matrices de confusión para ambos
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_dtree = confusion_matrix(y_test, y_pred_dtree)

print("=== Matriz de Confusión: k-NN (Datos Escalados) ===")
print(cm_knn)
print(f"Falsos Negativos (k-NN): {cm_knn[1, 0]}")
print(f"Falsos Positivos (k-NN): {cm_knn[0, 1]}\n")

print("=== Matriz de Confusión: Árbol de Decisión (Datos Sin Escalear) ===")
print(cm_dtree)
print(f"Falsos Negativos (Árbol): {cm_dtree[1, 0]}")
print(f"Falsos Positivos (Árbol): {cm_dtree[0, 1]}")

=== Matriz de Confusión: k-NN (Datos Escalados) ===
[[71  1]
 [ 4 38]]
Falsos Negativos (k-NN): 4
Falsos Positivos (k-NN): 1

=== Matriz de Confusión: Árbol de Decisión (Datos Sin Escalear) ===
[[70  2]
 [ 8 34]]
Falsos Negativos (Árbol): 8
Falsos Positivos (Árbol): 2


## Evaluación Teórica / Decisión de Negocios

1. **En este escenario médico, ¿que implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?**
   * Un Error Tipo II (Falso Negativo) significa que el modelo predice que un paciente tiene un tumor *Benigno* (0) cuando en realidad es *Maligno* (1).
   * Cuesta vidas porque el paciente se irá a casa creyendo que está sano, retrasando el inicio de un tratamiento oncológico oportuno. Para cuando el cáncer avance y se detecte nuevamente, podría encontrarse en una etapa avanzada o haber hecho metástasis, disminuyendo drásticamente las probabilidades de supervivencia.

2. **¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?**
   * Los Árboles de Decisión toman decisiones basadas en reglas de partición (umbrales) en una característica a la vez (por ejemplo: `¿feature_1 > 15.4?`). Al evaluar cada característica de forma independiente para hacer los cortes, la escala relativa de una variable con respecto a otra no altera la capacidad del árbol para encontrar el mejor punto de separación.
   * En cambio, algoritmos como k-NN dependen directamente del cálculo de distancias (como la Euclidiana). Si una característica tiene un rango de 0 a 1000 y otra de 0 a 1, la de mayor rango dominará por completo el cálculo de la distancia, anulando el efecto de la otra.